# 02. OHLC 60분봉 전처리와 +3% 라벨

같은 종목의 연속 60개 1분봉으로 모델 입력을 만들고, `t`봉 종료 후 `t+1 open` 진입을 가정해 미래 1·3·5분의 +3% 도달 여부와 MFE·MAE를 생성한다. 원본 데이터는 수정하지 않는다.

In [1]:
from pathlib import Path
from numpy.lib.stride_tricks import sliding_window_view
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root()
DATA_ROOT = (PROJECT_ROOT / '../../data/stock_data').resolve()
RESULTS_ROOT = (PROJECT_ROOT / '../../results').resolve()
OUTPUT_ROOT = RESULTS_ROOT / 'preprocessing'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

VERSION = 'ohlc_60m_tp3pct_v1'
SEQUENCE_LENGTH = 60
HORIZONS = [1, 3, 5]
MAX_HORIZON = max(HORIZONS)
TAKE_PROFIT_RETURN = 0.03
RAW_COLUMNS = ['symbol', 'timestamp_utc', 'open', 'high', 'low', 'close']

print('data   :', DATA_ROOT)
print('output :', OUTPUT_ROOT)
print('version:', VERSION)

data   : /home/user/urbandatalab/YSLee/data/stock_data
output : /home/user/urbandatalab/YSLee/results/preprocessing
version: ohlc_60m_tp3pct_v1


## 1. OHLC 적재와 유효 구간 분리

결측, OHLC 관계 오류, 중복 timestamp는 유효 구간에서 제외한다. 제외한 행 전후를 연결하지 않고 정확히 1분 간격인 행만 같은 run으로 묶는다.

In [2]:
files = sorted(DATA_ROOT.glob('raw/session_*/*_enriched.csv'))
assert files, f'enriched CSV가 없습니다: {DATA_ROOT}'

quality_rows = []
valid_parts = []
for file_number, path in enumerate(files):
    frame = pd.read_csv(path, usecols=RAW_COLUMNS, encoding='utf-8-sig')
    frame['symbol'] = frame['symbol'].astype('string').str.strip().str.upper()
    frame['timestamp_utc'] = pd.to_datetime(frame['timestamp_utc'], errors='coerce', utc=True)
    for column in ['open', 'high', 'low', 'close']:
        frame[column] = pd.to_numeric(frame[column], errors='coerce')
    frame = frame.sort_values(['symbol', 'timestamp_utc']).reset_index(drop=True)

    duplicate = frame.duplicated(['symbol', 'timestamp_utc'], keep=False)
    required_valid = frame[RAW_COLUMNS].notna().all(axis=1)
    valid_ohlc = (
        frame[['open', 'high', 'low', 'close']].gt(0).all(axis=1)
        & frame['high'].ge(frame[['open', 'close']].max(axis=1))
        & frame['low'].le(frame[['open', 'close']].min(axis=1))
        & frame['high'].ge(frame['low'])
    )
    valid = required_valid & valid_ohlc & ~duplicate
    clean = frame.loc[valid].copy()
    delta = clean.groupby('symbol')['timestamp_utc'].diff().dt.total_seconds().div(60)
    new_run = clean['symbol'].ne(clean['symbol'].shift()) | ~np.isclose(delta, 1.0, rtol=0.0, atol=1e-6)
    clean['run_number'] = new_run.cumsum().astype('int64')
    clean['run_id'] = path.stem + '::' + clean['run_number'].astype(str)
    clean['session'] = path.parent.name
    clean['source_file'] = str(path.resolve())
    valid_parts.append(clean)

    quality_rows.append({
        'session': path.parent.name,
        'source_file': str(path.resolve()),
        'raw_rows': len(frame),
        'required_missing_rows': int((~required_valid).sum()),
        'invalid_ohlc_rows': int((~valid_ohlc & required_valid).sum()),
        'duplicate_rows': int(duplicate.sum()),
        'valid_rows': int(valid.sum()),
        'continuous_runs': int(clean['run_id'].nunique()),
    })

data = pd.concat(valid_parts, ignore_index=True)
quality = pd.DataFrame(quality_rows)
print(f'files={len(files):,}, raw_rows={quality.raw_rows.sum():,}, valid_rows={len(data):,}')
display(quality.groupby('session')[['raw_rows', 'required_missing_rows', 'invalid_ohlc_rows', 'duplicate_rows', 'valid_rows', 'continuous_runs']].sum())

files=208, raw_rows=127,423, valid_rows=127,423


,raw_rows,required_missing_rows,invalid_ohlc_rows,duplicate_rows,valid_rows,continuous_runs
session,,,,,,
session_2026-07-07,7422,0,0,0,7422,865
session_2026-07-08,17218,0,0,0,17218,1348
session_2026-07-09,18787,0,0,0,18787,2026
session_2026-07-10,19458,0,0,0,19458,2154
session_2026-07-13,13024,0,0,0,13024,1122
session_2026-07-14,12995,0,0,0,12995,1175
session_2026-07-15,15364,0,0,0,15364,1031
session_2026-07-16,10051,0,0,0,10051,912
session_2026-07-17,13104,0,0,0,13104,1360


## 2. 60봉 sequence와 미래 라벨 생성

Sequence 특징은 OHLC만으로 만든다. `normalized OHLC`는 각 window 마지막 종가 기준 로그비율이다. 첫 봉의 `log_return_1`과 `gap_return`은 window 밖의 값을 참조하지 않고 0으로 둔다. 라벨용 `t+1 open`과 미래 high·low는 입력 sequence에 들어가지 않는다.

In [3]:
SEQUENCE_FEATURES = [
    'log_open_to_last_close', 'log_high_to_last_close',
    'log_low_to_last_close', 'log_close_to_last_close',
    'log_return_1', 'gap_return', 'body_return', 'range_return',
    'upper_wick', 'lower_wick', 'close_location',
]
CONTEXT_LOOKBACKS = [3, 5, 10, 20, 60]
CONTEXT_FEATURES = [
    f'{name}_{lookback}bars'
    for lookback in CONTEXT_LOOKBACKS
    for name in ['return', 'volatility', 'distance_from_high', 'distance_from_low']
]

def make_sequence_features(ohlc_windows):
    open_ = ohlc_windows[:, :, 0]
    high = ohlc_windows[:, :, 1]
    low = ohlc_windows[:, :, 2]
    close = ohlc_windows[:, :, 3]
    reference = close[:, -1:]
    normalized = np.log(ohlc_windows / reference[:, :, None])

    log_return = np.zeros_like(close)
    log_return[:, 1:] = np.log(close[:, 1:] / close[:, :-1])
    gap_return = np.zeros_like(close)
    gap_return[:, 1:] = open_[:, 1:] / close[:, :-1] - 1.0
    body = close / open_ - 1.0
    range_ = (high - low) / open_
    upper_wick = (high - np.maximum(open_, close)) / open_
    lower_wick = (np.minimum(open_, close) - low) / open_
    close_location = np.divide(
        close - low, high - low, out=np.full_like(close, 0.5), where=(high > low),
    )
    derived = np.stack([log_return, gap_return, body, range_, upper_wick, lower_wick, close_location], axis=2)
    return np.concatenate([normalized, derived], axis=2).astype('float32')

def make_context_features(ohlc_windows, sequence_features):
    open_ = ohlc_windows[:, :, 0]
    high = ohlc_windows[:, :, 1]
    low = ohlc_windows[:, :, 2]
    close = ohlc_windows[:, :, 3]
    log_return = sequence_features[:, :, SEQUENCE_FEATURES.index('log_return_1')]
    context = []
    for lookback in CONTEXT_LOOKBACKS:
        recent_open = open_[:, -lookback:]
        recent_high = high[:, -lookback:]
        recent_low = low[:, -lookback:]
        recent_close = close[:, -lookback:]
        last_close = recent_close[:, -1]
        context.extend([
            last_close / recent_open[:, 0] - 1.0,
            log_return[:, -lookback:].std(axis=1),
            last_close / recent_high.max(axis=1) - 1.0,
            last_close / recent_low.min(axis=1) - 1.0,
        ])
    return np.column_stack(context).astype('float32')

def build_run_samples(run):
    run = run.sort_values('timestamp_utc').reset_index(drop=True)
    sample_count = len(run) - SEQUENCE_LENGTH - MAX_HORIZON + 1
    if sample_count <= 0:
        return None
    ohlc = run[['open', 'high', 'low', 'close']].to_numpy(dtype='float64')
    windows = sliding_window_view(ohlc, SEQUENCE_LENGTH, axis=0).transpose(0, 2, 1)[:sample_count]
    sequence = make_sequence_features(windows)
    context = make_context_features(windows, sequence)
    ends = np.arange(SEQUENCE_LENGTH - 1, SEQUENCE_LENGTH - 1 + sample_count)
    entry = ohlc[ends + 1, 0]
    metadata = pd.DataFrame({
        'session': run.loc[ends, 'session'].to_numpy(),
        'symbol': run.loc[ends, 'symbol'].to_numpy(),
        'source_file': run.loc[ends, 'source_file'].to_numpy(),
        'run_id': run.loc[ends, 'run_id'].to_numpy(),
        'input_start_timestamp': run.loc[ends - SEQUENCE_LENGTH + 1, 'timestamp_utc'].to_numpy(),
        'decision_timestamp': run.loc[ends, 'timestamp_utc'].to_numpy(),
        'entry_timestamp': run.loc[ends + 1, 'timestamp_utc'].to_numpy(),
        'decision_close': ohlc[ends, 3],
        'entry_open': entry,
    })
    for horizon in HORIZONS:
        future_high = np.column_stack([ohlc[ends + minute, 1] for minute in range(1, horizon + 1)])
        future_low = np.column_stack([ohlc[ends + minute, 2] for minute in range(1, horizon + 1)])
        mfe = future_high.max(axis=1) / entry - 1.0
        mae = future_low.min(axis=1) / entry - 1.0
        metadata[f'mfe_{horizon}m'] = mfe.astype('float32')
        metadata[f'mae_{horizon}m'] = mae.astype('float32')
        metadata[f'target_tp3_{horizon}m'] = (mfe >= TAKE_PROFIT_RETURN).astype('int8')
    future_high_5m = np.column_stack([ohlc[ends + minute, 1] for minute in range(1, MAX_HORIZON + 1)])
    hit = future_high_5m >= entry[:, None] * (1.0 + TAKE_PROFIT_RETURN)
    metadata['time_to_tp3_5m'] = np.where(hit.any(axis=1), hit.argmax(axis=1) + 1, 0).astype('int8')
    return metadata, sequence, context

print('sequence features:', len(SEQUENCE_FEATURES))
print('context features :', len(CONTEXT_FEATURES))

sequence features: 11
context features : 20


In [4]:
metadata_parts = []
sequence_parts = []
context_parts = []
for _, run in data.groupby(['source_file', 'symbol', 'run_id'], sort=False):
    result = build_run_samples(run)
    if result is None:
        continue
    metadata_part, sequence_part, context_part = result
    metadata_parts.append(metadata_part)
    sequence_parts.append(sequence_part)
    context_parts.append(context_part)

metadata = pd.concat(metadata_parts, ignore_index=True)
metadata.insert(0, 'sample_id', np.arange(len(metadata), dtype='int64'))
sequence = np.concatenate(sequence_parts).astype('float32')
context = np.concatenate(context_parts).astype('float32')

assert sequence.shape == (len(metadata), SEQUENCE_LENGTH, len(SEQUENCE_FEATURES))
assert context.shape == (len(metadata), len(CONTEXT_FEATURES))
assert np.isfinite(sequence).all() and np.isfinite(context).all()
assert metadata[['mfe_1m', 'mfe_3m', 'mfe_5m', 'mae_1m', 'mae_3m', 'mae_5m']].notna().all().all()
assert metadata['sample_id'].is_unique
assert metadata['decision_timestamp'].sub(metadata['input_start_timestamp']).eq(pd.Timedelta(minutes=59)).all()
assert metadata['entry_timestamp'].sub(metadata['decision_timestamp']).eq(pd.Timedelta(minutes=1)).all()
assert metadata['target_tp3_1m'].le(metadata['target_tp3_3m']).all()
assert metadata['target_tp3_3m'].le(metadata['target_tp3_5m']).all()
assert metadata['mfe_1m'].le(metadata['mfe_3m'] + 1e-7).all()
assert metadata['mfe_3m'].le(metadata['mfe_5m'] + 1e-7).all()
assert metadata['mae_1m'].ge(metadata['mae_3m'] - 1e-7).all()
assert metadata['mae_3m'].ge(metadata['mae_5m'] - 1e-7).all()

print('metadata:', metadata.shape)
print('sequence:', sequence.shape, sequence.dtype)
print('context :', context.shape, context.dtype)

metadata: (64144, 20)
sequence: (64144, 60, 11) float32
context : (64144, 20) float32


## 3. 라벨 분포 확인

전체 평균뿐 아니라 날짜별 양성률을 확인한다. 아직 Train/Test 날짜는 확정하지 않는다.

In [5]:
label_columns = [f'target_tp3_{horizon}m' for horizon in HORIZONS]
distribution = pd.DataFrame({
    'samples': [len(metadata)] * len(HORIZONS),
    'positives': [int(metadata[column].sum()) for column in label_columns],
    'positive_rate': [float(metadata[column].mean()) for column in label_columns],
    'mfe_median': [float(metadata[f'mfe_{horizon}m'].median()) for horizon in HORIZONS],
    'mae_median': [float(metadata[f'mae_{horizon}m'].median()) for horizon in HORIZONS],
}, index=[f'{horizon}m' for horizon in HORIZONS])
display(distribution.style.format({'positive_rate': '{:.3%}', 'mfe_median': '{:.3%}', 'mae_median': '{:.3%}'}))

session_distribution = metadata.groupby('session')[label_columns].agg(['size', 'sum', 'mean'])
display(session_distribution)
display(metadata[['sample_id', 'session', 'symbol', 'input_start_timestamp', 'decision_timestamp', 'entry_timestamp', 'entry_open', *label_columns, 'mfe_5m', 'mae_5m', 'time_to_tp3_5m']].head())

,samples,positives,positive_rate,mfe_median,mae_median
1m,64144,2775,4.326%,0.329%,-0.344%
3m,64144,7619,11.878%,0.691%,-0.696%
5m,64144,11074,17.264%,0.881%,-0.901%


target_tp3_1m                target_tp3_3m                  \
                            size  sum      mean          size   sum      mean   
session                                                                         
session_2026-07-07          3781   87  0.023010          3781   275  0.072732   
session_2026-07-08          9147  321  0.035093          9147  1082  0.118290   
session_2026-07-09          9038  327  0.036181          9038   911  0.100797   
session_2026-07-10          9077  622  0.068525          9077  1497  0.164922   
session_2026-07-13          7297  376  0.051528          7297   947  0.129779   
session_2026-07-14          6120  253  0.041340          6120   682  0.111438   
session_2026-07-15          8587  444  0.051706          8587  1168  0.136020   
session_2026-07-16          5338  207  0.038779          5338   623  0.116710   
session_2026-07-17          5759  138  0.023962          5759   434  0.075360   

                   target_tp3_5m                  
                            size   sum      mean  
session                                           
session_2026-07-07          3781   407  0.107643  
session_2026-07-08          9147  1646  0.179950  
session_2026-07-09          9038  1356  0.150033  
session_2026-07-10          9077  2103  0.231684  
session_2026-07-13          7297  1346  0.184459  
session_2026-07-14          6120   978  0.159804  
session_2026-07-15          8587  1677  0.195295  
session_2026-07-16          5338   898  0.168228  
session_2026-07-17          5759   663  0.115124

,sample_id,session,symbol,input_start_timestamp,decision_timestamp,entry_timestamp,entry_open,target_tp3_1m,target_tp3_3m,target_tp3_5m,mfe_5m,mae_5m,time_to_tp3_5m
0,0,session_2026-07-07,ALM,2026-07-07 13:25:00+00:00,2026-07-07 14:24:00+00:00,2026-07-07 14:25:00+00:00,14.920,0,0,0,0.008378,-0.004692,0
1,1,session_2026-07-07,ALM,2026-07-07 13:26:00+00:00,2026-07-07 14:25:00+00:00,2026-07-07 14:26:00+00:00,14.910,0,0,0,0.013414,-0.004024,0
2,2,session_2026-07-07,ALM,2026-07-07 13:27:00+00:00,2026-07-07 14:26:00+00:00,2026-07-07 14:27:00+00:00,14.855,0,0,0,0.017503,0.000000,0
3,3,session_2026-07-07,ALM,2026-07-07 13:28:00+00:00,2026-07-07 14:27:00+00:00,2026-07-07 14:28:00+00:00,14.960,0,0,0,0.010361,-0.002340,0
4,4,session_2026-07-07,ALM,2026-07-07 13:29:00+00:00,2026-07-07 14:28:00+00:00,2026-07-07 14:29:00+00:00,14.985,0,0,0,0.008675,-0.004151,0


## 4. Artifact 저장

Metadata와 라벨은 Parquet, 모델 입력 배열은 압축 NPZ, 특징 정의는 JSON으로 저장한다. 스케일링은 날짜 split 확정 후 Train 구간에서만 학습해야 하므로 현재 수행하지 않는다.

In [6]:
paths = {
    'metadata': OUTPUT_ROOT / f'{VERSION}_metadata.parquet',
    'features': OUTPUT_ROOT / f'{VERSION}_features.npz',
    'quality': OUTPUT_ROOT / f'{VERSION}_quality.parquet',
    'schema': OUTPUT_ROOT / f'{VERSION}_schema.json',
}
metadata.to_parquet(paths['metadata'], index=False, compression='zstd')
quality.to_parquet(paths['quality'], index=False, compression='zstd')
np.savez_compressed(paths['features'], sequence=sequence, context=context)
schema = {
    'version': VERSION,
    'raw_model_columns': ['open', 'high', 'low', 'close'],
    'sequence_length': SEQUENCE_LENGTH,
    'sequence_features': SEQUENCE_FEATURES,
    'context_features': CONTEXT_FEATURES,
    'decision_rule': 'after completed t bar',
    'entry_proxy': 'open[t+1]',
    'take_profit_return': TAKE_PROFIT_RETURN,
    'horizons_minutes': HORIZONS,
    'requires_consecutive_input_and_label_minutes': True,
    'scaling_applied': False,
}
paths['schema'].write_text(json.dumps(schema, ensure_ascii=False, indent=2), encoding='utf-8')

artifact_summary = pd.DataFrame([
    {'artifact': name, 'path': str(path), 'size_mb': path.stat().st_size / 1024**2}
    for name, path in paths.items()
])
display(artifact_summary)

,artifact,path,size_mb
0,metadata,/home/user/urbandatalab/YSLee/results/preproce...,2.194470
1,features,/home/user/urbandatalab/YSLee/results/preproce...,35.248597
2,quality,/home/user/urbandatalab/YSLee/results/preproce...,0.009660
3,schema,/home/user/urbandatalab/YSLee/results/preproce...,0.001178


## 현재 단계 결론

연속 60봉 OHLC 입력과 +3% 도달·MFE·MAE 라벨을 생성했다. 다음 단계에서는 날짜별 양성률과 겹치는 window 수를 기준으로 주 horizon, Train/Test 날짜, 학습 sampling 간격을 확정한다. Test 결과를 본 뒤 이 기준을 바꾸지 않는다.